# KAT-VAD v2 — P3 / **E1: rate-matched DoTA evaluation** (no training)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§8 (Amendment 2, D10–D11, J1–J9)** · Plan: `.project/plans/katvad-v2-e0-e2.md` P3.
Re-scores the **3 phase-4 KIP-off checkpoints** (seeds 2024–2026) on **DoTA-dev (702 clips)** from `DoTA_s1_ncc`:

| arm | features | sequence |
|---|---|---|
| **A** | `s1[::8]` (0.8 s/step, today's protocol) | whole clip |
| **B** | `s1[::3]` (0.3 s/step, T2's rate) | whole clip |
| **C** | `s1[::3]` | windows W = 20, hop 4, overlap-averaged |

All arms are interpolated to native frames and scored against the same native labels. Decision (J7): B or C is adopted
only if its paired Δ vs A has a **cluster-bootstrap lower bound > 0**; otherwise A stays.

| step | what | GPU? |
|---|---|---|
| 1 | preflight: tree has Amendment 2, checkpoints + phase-4 results + s1 cache exist | no |
| 2 | stage DoTA-dev s1 features to local disk | no |
| 3 | run E1 (A → **J9 regression gate** → B, C → read-out) | yes (T4 is plenty) |
| 4 | record | no |

**Do not change a flag after reading §3.** The rule was committed before the first number. DoTA-eval is never loaded.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE = '/content/drive/MyDrive/Thesis'
os.environ['PROJECT_ROOT']       = DRIVE
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE}/outputs'

os.environ['REPO'] = f'{DRIVE}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
sys.path.insert(0, str(REPO))

# core.constants reads KATVAD_* at import time -- import AFTER the block above.
from core import constants  # noqa: E402

SEEDS    = (2024, 2025, 2026)
S1_DIR   = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
DOTA     = constants.DATA_ROOT / 'DoTA'
LABELS   = DOTA / 'labels_s8'                       # only defs.json is read from here
REPORT   = constants.OUTPUT_ROOT / 'REPORTS' / 'v2_E1'
LOCAL_S1 = Path('/content/e1/DoTA_s1_ncc_dev')      # VM-local disk, never Drive
REPORT.mkdir(parents=True, exist_ok=True)

# phase-4 runs: Drive layout is outputs/DADA2000_orig_phase4/s{seed}/...
candidates = [constants.OUTPUT_ROOT / 'DADA2000_orig_phase4',
              constants.OUTPUT_ROOT / 'v1' / 'DADA2000_orig_phase4']
RUNS = next((p for p in candidates if p.is_dir()), None)
assert RUNS is not None, f'no phase-4 run dir among {candidates} -- set RUNS by hand'
print('RUNS', RUNS)

In [ ]:
%%bash
# transformers pinned like phase 4 (C7/C21).
pip install -q "transformers==4.56.*"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU (CPU works, ~10x slower)'

## 1. Preflight — refuse to start on a tree or data that E1 was not registered on

In [ ]:
import json
import subprocess

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert 'Amendment 2' in addendum and 'J9' in addendum, (
    'this upload predates Amendment 2 -- re-upload branch v2')
from core.tools import rate_matched_eval  # noqa: E402  (fails fast if absent)
from core.data.v2_splits import load_split  # noqa: E402

dev = load_split(constants.V2_SPLIT_DOTA_DEV)        # sha1-checked against the frozen manifest
assert len(dev) == 702, len(dev)

RUN_ARGS = []
for seed in SEEDS:
    ckpt = RUNS / f's{seed}' / 'stage2_kip_off' / 'checkpoint_last.pt'
    res = RUNS / f's{seed}' / 'eval_dota_kip_off' / 'results.json'
    assert ckpt.is_file(), f'missing {ckpt}'
    assert res.is_file(), f'missing {res}'
    old = json.loads(res.read_text())
    assert all(v in old['per_video'] for v in dev), f'{res} lacks some DoTA-dev clips'
    RUN_ARGS += ['--run', f's{seed}', str(ckpt), str(res)]
    print(f's{seed}: ckpt OK | phase-4 DoTA macro (full set, s8 labels) {old["auc_macro"]:.4f}')

for f in (DOTA / 'metadata_val.json', DOTA / 'val_split.txt', LABELS / constants.DEFS_FILENAME):
    assert f.is_file(), f'missing {f}'
missing = [v for v in dev if not (S1_DIR / f'{v}.npy').exists()]
assert not missing, f'{len(missing)} DoTA-dev clips have no s1 feature: {missing[:5]}'
print(f'DoTA-dev {len(dev)} clips, all present in {S1_DIR}')

## 2. Stage DoTA-dev s1 features to local disk (only the 702 dev clips; ~150 MB)

Drive FUSE reads are slow and the harness reads each file once per run; staging makes it one read.
A spot check re-verifies `s1[::8] == DoTA_s8_ncc` on the staged copy.

In [ ]:
import shutil
import numpy as np

LOCAL_S1.mkdir(parents=True, exist_ok=True)
for i, v in enumerate(dev):
    dst = LOCAL_S1 / f'{v}.npy'
    if not dst.exists():
        shutil.copy2(S1_DIR / f'{v}.npy', dst)
    if i % 100 == 0:
        print(f'{i}/{len(dev)}')
S8_DIR = constants.CLIP_CACHE_DIR / 'DoTA_s8_ncc'
if S8_DIR.is_dir():
    for v in dev[:: len(dev) // 20]:
        a, b = np.load(LOCAL_S1 / f'{v}.npy')[::8], np.load(S8_DIR / f'{v}.npy')
        assert a.shape == b.shape and np.allclose(a, b, atol=1e-4), f'{v}: s1[::8] != s8'
    print('spot check s1[::8] == s8: OK')
print(f'staged {len(list(LOCAL_S1.glob("*.npy")))} files -> {LOCAL_S1}')

## 3. Run E1

Order inside the tool: arm A for all three seeds → **J9 regression gate** (A at step level must reproduce each
seed's phase-4 `max_score` on every dev clip, atol 1e-4; a failure **raises and nothing else is scored**) → arms B, C
→ read-out. Expect a few minutes on a GPU.

If J9 fails: **stop**. Do not loosen the tolerance; send the error to Claude.

In [ ]:
cmd = [sys.executable, '-m', 'core.tools.rate_matched_eval', *RUN_ARGS,
       '--s1-dir', LOCAL_S1, '--metadata', DOTA / 'metadata_val.json',
       '--split-file', DOTA / 'val_split.txt', '--data-dir', LABELS,
       '--out-dir', REPORT, '--device', 'auto']
subprocess.run([str(a) for a in cmd], cwd=str(REPO), check=True)
print((REPORT / rate_matched_eval.READOUT_MD).read_text())

## 4. Record

`e1_readout.{json,md}` are on Drive under `outputs/REPORTS/v2_E1/`. Paste the markdown back to Claude. Then:

* **adopt A** → DoTA protocol stays s8 whole clip; P2 (E2) stands as is.
* **adopt B or C** → D11: E2(b) and E2(c) are re-read on DoTA-dev at stride 3 before P5 fixes the CRN reference
  (Claude ships that runbook next).